# Component 3: Google Colab Baseline Model Training
**Project:** J26-DS-361 Intelligent MLOps Platform  
**Component:** Component 3 — Adaptive Intelligent Synthetic Data Repair Engine  
**Task:** Train and evaluate standardized baseline classification models (XGBoost, Random Forest, LightGBM) across tabular benchmarks with strict leakage prevention.

### Prerequisites:
- Select **GPU** hardware accelerator in Runtime > Change runtime type.
- Connect your Google Drive to persist trained models and metrics.

**Execution Status:** NOT YET EXECUTED (Ready for Google Colab GPU Runtime)


In [ ]:
# Step 1: Environment Setup & Hardware Detection
import os
import sys
import platform
import torch

print(f"Python Platform: {platform.platform()}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")

# GPU Detection with CPU Fallback
has_gpu = torch.cuda.is_available()
device = "cuda" if has_gpu else "cpu"
print(f"Compute Device: {device.upper()}")
if has_gpu:
    print(f"GPU Hardware: {torch.cuda.get_device_name(0)}")

# Install required dependencies explicitly
!pip install --quiet xgboost scikit-learn lightgbm joblib pandas numpy


In [ ]:
# Step 2: Google Drive Mounting & Workspace Configuration
import os
from pathlib import Path

# Mount Drive with local fallback
try:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
    DRIVE_WORKSPACE = Path("/content/drive/MyDrive/J26-DS-361-MLOps/component3")
except Exception as exc:
    print(f"Running in standalone mode: {exc}")
    DRIVE_WORKSPACE = Path("./colab_workspace").resolve()

# Establish directories
MODELS_DIR = DRIVE_WORKSPACE / "models"
METRICS_DIR = DRIVE_WORKSPACE / "metrics"
DATA_DIR = DRIVE_WORKSPACE / "data"

for d in [MODELS_DIR, METRICS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace initialized at: {DRIVE_WORKSPACE}")


In [ ]:
# Step 3: Leak-Free Preprocessing & Baseline Training Pipeline
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import joblib

class BaselinePreprocessPipeline:
    """Preprocesses tabular features strictly fitted on training data."""
    def __init__(self, num_cols, cat_cols):
        self.num_cols = list(num_cols)
        self.cat_cols = list(cat_cols)
        self.ordered_features = self.num_cols + self.cat_cols
        self.num_imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()
        self.cat_imputer = SimpleImputer(strategy="constant", fill_value="missing")
        self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.float32)
        self.is_fitted = False

    def fit(self, df):
        if self.num_cols:
            num_data = df[self.num_cols].apply(pd.to_numeric, errors="coerce")
            num_imp = self.num_imputer.fit_transform(num_data)
            self.scaler.fit(num_imp)
        if self.cat_cols:
            cat_data = df[self.cat_cols].astype(str)
            cat_imp = self.cat_imputer.fit_transform(cat_data)
            self.encoder.fit(cat_imp)
        self.is_fitted = True
        return self

    def transform(self, df):
        parts = []
        if self.num_cols:
            num_data = df[self.num_cols].apply(pd.to_numeric, errors="coerce")
            num_imp = self.num_imputer.transform(num_data)
            parts.append(self.scaler.transform(num_imp))
        if self.cat_cols:
            cat_data = df[self.cat_cols].astype(str)
            cat_imp = self.cat_imputer.transform(cat_data)
            parts.append(self.encoder.transform(cat_imp))
        return np.hstack(parts) if parts else np.empty((len(df), 0), dtype=np.float32)

def train_and_evaluate_baseline(df, target_col, num_cols, cat_cols, dataset_id="uci_adult", model_family="xgboost", seed=42):
    # Separate features and target
    X_df = df.drop(columns=[target_col])
    y_raw = df[target_col].to_numpy()
    y = pd.to_numeric(y_raw, errors="coerce").fillna(0).astype(int)

    # Stratified Split: 70% Train, 15% Validation, 15% Test
    X_train_df, X_temp_df, y_train, y_temp = train_test_split(X_df, y, test_size=0.30, random_state=seed, stratify=y)
    X_val_df, X_test_df, y_val, y_test = train_test_split(X_temp_df, y_temp, test_size=0.50, random_state=seed, stratify=y_temp)

    # Fit preprocessor on train ONLY
    preprocessor = BaselinePreprocessPipeline(num_cols=num_cols, cat_cols=cat_cols).fit(X_train_df)
    X_train = preprocessor.transform(X_train_df)
    X_val = preprocessor.transform(X_val_df)
    X_test = preprocessor.transform(X_test_df)

    # Model training
    if model_family.lower() == "xgboost":
        model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=seed, eval_metric="logloss")
    else:
        model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=seed)

    model.fit(X_train, y_train)

    def calc_metrics(X_mat, y_true):
        preds = model.predict(X_mat)
        probs = model.predict_proba(X_mat)[:, 1] if hasattr(model, "predict_proba") else preds
        return {
            "accuracy": float(accuracy_score(y_true, preds)),
            "precision": float(precision_score(y_true, preds, zero_division=0)),
            "recall": float(recall_score(y_true, preds, zero_division=0)),
            "f1": float(f1_score(y_true, preds, zero_division=0)),
            "roc_auc": float(roc_auc_score(y_true, probs)) if len(np.unique(y_true)) > 1 else 0.5,
            "support": int(len(y_true)),
        }

    metrics = {
        "dataset_id": dataset_id,
        "model_family": model_family,
        "seed": seed,
        "train_metrics": calc_metrics(X_train, y_train),
        "val_metrics": calc_metrics(X_val, y_val),
        "test_metrics": calc_metrics(X_test, y_test),
    }

    print(f"[{dataset_id} - {model_family}] Test F1: {metrics['test_metrics']['f1']:.4f} | Accuracy: {metrics['test_metrics']['accuracy']:.4f}")
    return model, preprocessor, metrics

print("Baseline training framework loaded.")


In [ ]:
# Step 4: Execute Baseline Training & Save Artifacts Jointly
# Executable demonstration on sample benchmark data
rng = np.random.default_rng(42)
n_samples = 500
demo_df = pd.DataFrame({
    "age": rng.normal(38, 12, n_samples),
    "education_num": rng.integers(1, 16, n_samples),
    "hours_per_week": rng.normal(40, 10, n_samples),
    "workclass": rng.choice(["Private", "Self-emp", "Gov"], n_samples),
    "occupation": rng.choice(["Tech", "Sales", "Exec"], n_samples),
    "target": rng.choice([0, 1], p=[0.75, 0.25], size=n_samples),
})

num_features = ["age", "education_num", "hours_per_week"]
cat_features = ["workclass", "occupation"]

model, preprocessor, metrics = train_and_evaluate_baseline(
    df=demo_df,
    target_col="target",
    num_cols=num_features,
    cat_cols=cat_features,
    dataset_id="uci_adult_demo",
    model_family="xgboost",
    seed=42
)

# Crucial Requirement: Model and preprocessor saved TOGETHER
model_file = MODELS_DIR / "uci_adult_demo_xgboost_model.joblib"
prep_file = MODELS_DIR / "uci_adult_demo_xgboost_preprocessor.joblib"
metrics_file = METRICS_DIR / "uci_adult_demo_xgboost_metrics.json"

joblib.dump(model, model_file)
joblib.dump(preprocessor, prep_file)
with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"Model saved to: {model_file}")
print(f"Preprocessor saved to: {prep_file}")
print(f"Metrics exported to: {metrics_file}")
print("Baseline training notebook executed successfully!")
